<a href="https://colab.research.google.com/github/lorenzobalzani/nlp_assigments/blob/master/NLP_Project/NLP_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Dobbiamo paddare input per lunghezza massima?
2. L'input del modello (encoder + classifier) è la concatenazione degli Utturances per ogni istanza o una Utturances per volta

Idea: classification head for emotions is jutst a normal linear classification (emotions are not relative to the utterance), instead for triggers needs to be an LSTM. (triggers are relative to the utterance).



In [40]:
import gdown
import json
import pandas as pd

from transformers import BertModel, BertTokenizer, AutoTokenizer, AutoModel
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.nn.utils.rnn import pad_sequence

In [3]:
training_set_filename = f"MELD_train_efr.json"

try:
  print(f"Attempting to download {training_set_filename} from Drive...")
  file_id = '1wVNU2XvvhqjaGXZM-JLJwOt97gt4g9j2'  # Extracted File ID from your provided URL
  url = f'https://drive.google.com/uc?id={file_id}'
  gdown.download(url, training_set_filename, quiet=False)
  with open(training_set_filename, 'r') as file:
        training_set_json = json.load(file)

  print(f"\nSuccessfully downloaded {training_set_filename} from Drive!")
except:
  print(f"Error loading {training_set_filename} set from Drive")

Attempting to download MELD_train_efr.json from Drive...


Downloading...
From: https://drive.google.com/uc?id=1wVNU2XvvhqjaGXZM-JLJwOt97gt4g9j2
To: /content/MELD_train_efr.json
100%|██████████| 5.18M/5.18M [00:00<00:00, 205MB/s]


Successfully downloaded MELD_train_efr.json from Drive!


In [4]:
print(f"Training set has length: {len(training_set_json)}")

Training set has length: 4000


In [5]:
for i in [0,100,2000]:
  print(training_set_json[i])

{'episode': 'utterance_0', 'speakers': ['Chandler', 'The Interviewer', 'Chandler', 'The Interviewer', 'Chandler'], 'emotions': ['neutral', 'neutral', 'neutral', 'neutral', 'surprise'], 'utterances': ["also I was the point person on my company's transition from the KL-5 to GR-6 system.", "You must've had your hands full.", 'That I did. That I did.', "So let's talk a little bit about your duties.", 'My duties?  All right.'], 'triggers': [0.0, 0.0, 0.0, 1.0, 0.0]}
{'episode': 'utterance_100', 'speakers': ['Mr. Treeger', 'Joey', 'Mr. Treeger', 'Joey', 'Mr. Treeger', 'Joey', 'Mr. Treeger', 'Joey', 'Mr. Treeger', 'Joey', 'Mr. Treeger', 'Joey', 'Mr. Treeger', 'Joey'], 'emotions': ['joy', 'joy', 'joy', 'joy', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'neutral', 'surprise', 'neutral', 'neutral'], 'utterances': [': I know, we did it!! Hey, that was incredible, huh?!', 'I know, it was amazing! I mean, we totally nailed it, it was beautiful.', ': Thank you, listen, thanks a

Let's turn the dict into a dataframe for convenience.

In [6]:
training_set_df = pd.DataFrame(training_set_json)
training_set_df.head()

,episode,speakers,emotions,utterances,triggers
0,utterance_0,"[Chandler, The Interviewer, Chandler, The Inte...","[neutral, neutral, neutral, neutral, surprise]",[also I was the point person on my company's t...,"[0.0, 0.0, 0.0, 1.0, 0.0]"
1,utterance_1,"[Chandler, The Interviewer, Chandler, The Inte...","[neutral, neutral, neutral, neutral, surprise,...",[also I was the point person on my company's t...,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]"
2,utterance_2,"[Chandler, The Interviewer, Chandler, The Inte...","[neutral, neutral, neutral, neutral, surprise,...",[also I was the point person on my company's t...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ..."
3,utterance_3,"[Chandler, The Interviewer, Chandler, The Inte...","[neutral, neutral, neutral, neutral, surprise,...",[also I was the point person on my company's t...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,utterance_4,"[Joey, Rachel, Joey, Rachel]","[surprise, sadness, surprise, fear]",[But then who? The waitress I went out with la...,"[0.0, 0.0, 1.0, 0.0]"


Let's remove NaN values in the triggers lists.

In [7]:
# Check the number of NaN values within the lists in the 'triggers' column
nan_count_before = training_set_df['triggers'].apply(lambda x: pd.Series(x).isna().sum()).sum()

# Display the number of NaN values
print(f"Number of NaN values in the 'triggers' column lists: {nan_count_before}")

# Replace NaN values within each list in the 'triggers' column with 0.0
training_set_df['triggers'] = training_set_df['triggers'].apply(lambda x: [0.0 if pd.isna(val) else val for val in x])

# Check again the number of NaN values within the lists in the 'triggers' column
nan_count_after = training_set_df['triggers'].apply(lambda x: pd.Series(x).isna().sum()).sum()

# Display the number of NaN values after replacing with 0s
print(f"Number of NaN values in the 'triggers' column lists after replacing: {nan_count_after}")

Number of NaN values in the 'triggers' column lists: 9
Number of NaN values in the 'triggers' column lists after replacing: 0


In [8]:
training_set_df.head()

,episode,speakers,emotions,utterances,triggers
0,utterance_0,"[Chandler, The Interviewer, Chandler, The Inte...","[neutral, neutral, neutral, neutral, surprise]",[also I was the point person on my company's t...,"[0.0, 0.0, 0.0, 1.0, 0.0]"
1,utterance_1,"[Chandler, The Interviewer, Chandler, The Inte...","[neutral, neutral, neutral, neutral, surprise,...",[also I was the point person on my company's t...,"[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0]"
2,utterance_2,"[Chandler, The Interviewer, Chandler, The Inte...","[neutral, neutral, neutral, neutral, surprise,...",[also I was the point person on my company's t...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, ..."
3,utterance_3,"[Chandler, The Interviewer, Chandler, The Inte...","[neutral, neutral, neutral, neutral, surprise,...",[also I was the point person on my company's t...,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,utterance_4,"[Joey, Rachel, Joey, Rachel]","[surprise, sadness, surprise, fear]",[But then who? The waitress I went out with la...,"[0.0, 0.0, 1.0, 0.0]"


## Process data

In [12]:
labels_emotions = [label for label in training_set_df['emotions'].explode().unique()]
id2label_emotions = {idx:label for idx, label in enumerate(labels_emotions)}
label2id_emotions = {label:idx for idx, label in enumerate(labels_emotions)}
labels_emotions

['neutral', 'surprise', 'fear', 'sadness', 'joy', 'disgust', 'anger']

In [13]:
# Let's define the maximum length of each utterance for determining padding max_length
max_length_sentences = max(training_set_df['utterances'].str.join(" ").str.split(" ").str.len())
print(f"The longest sentence is {max_length_sentences} words long.")
max_length_utturances = max(training_set_df['utterances'].str.len())
print(f"The longest utturance is {max_length_utturances} sentences long.")

The longest sentence is 266 words long.
The longest utturance is 24 sentences long.


In [38]:


def prepare_data(df, label_col='emotions', utterance_col='utterances', max_length_sentences=max_length_sentences):
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for _, row in df.iterrows():
        utterance_input_ids = []
        utterance_attention_masks = []

        for sentence in row[utterance_col]:
            encoded_dict = tokenizer.encode_plus(
                sentence,
                add_special_tokens=True,
                max_length=max_length_sentences,
                padding="max_length",
                return_attention_mask=True,
                return_tensors='pt',
            )
            utterance_input_ids.append(encoded_dict['input_ids'].squeeze(0))
            utterance_attention_masks.append(encoded_dict['attention_mask'].squeeze(0))


        all_input_ids.append(utterance_input_ids)
        all_attention_masks.append(utterance_attention_masks)
        all_labels.append([label2id_emotions[label] for label in row[label_col]])

    return all_input_ids, all_attention_masks, all_labels


# Prepare the data
input_ids, attention_masks, labels = prepare_data(training_set_df)

# Splits
train_inputs, test_inputs, train_labels, test_labels = train_test_split(input_ids, labels, random_state=42, test_size=0.1)

train_masks, test_masks, _, _ = train_test_split(attention_masks, input_ids, random_state=42, test_size=0.1)
train_masks, validation_masks, _, _ = train_test_split(train_masks, train_inputs, random_state=42, test_size=float(1/9))

train_inputs, validation_inputs, train_labels, validation_labels = train_test_split(train_inputs, train_labels, random_state=42, test_size=float(1/9))


In [56]:
# Prepare the training data set
train_data = list(zip(train_inputs, train_masks, train_labels))

# Prepare the validation data set
validation_data = list(zip(validation_inputs, validation_masks, validation_labels))

# Prepare the test data set
test_data = list(zip(test_inputs, test_masks, test_labels))


from torch.nn.utils.rnn import pad_sequence
import torch

def custom_collate_fn(batch):
    batch_input_ids, batch_attention_masks, batch_labels = zip(*batch)

    # Flatten input_ids and attention_masks
    flattened_input_ids = [item for sublist in batch_input_ids for item in sublist]
    flattened_attention_masks = [item for sublist in batch_attention_masks for item in sublist]

    # Pad the flattened sequences for this batch
    padded_input_ids = pad_sequence(flattened_input_ids, batch_first=True, padding_value=0)
    padded_attention_masks = pad_sequence(flattened_attention_masks, batch_first=True, padding_value=0)

    # Flatten the labels and convert to a tensor
    flattened_labels = [label for sublist in batch_labels for label in sublist]
    batch_labels_tensor = torch.tensor(flattened_labels, dtype=torch.long)

    return padded_input_ids, padded_attention_masks, batch_labels_tensor



from torch.utils.data import DataLoader

# Assuming train_data, validation_data, etc., are lists or similar structures containing your data
train_dataloader = DataLoader(train_data, batch_size=1, collate_fn=custom_collate_fn)
validation_dataloader = DataLoader(validation_data, batch_size=1, collate_fn=custom_collate_fn)
test_dataloader = DataLoader(test_data, batch_size=1, collate_fn=custom_collate_fn)


In [54]:
print(len(train_data))
print(len(validation_data))
print(len(test_data))

3200
400
400


Sanity checks

In [60]:
from transformers import BertModel, BertTokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F

class BertLSTMClassifier(nn.Module):
    def __init__(self, bert_model, hidden_size, num_classes, freeze_bert=False):
        super(BertLSTMClassifier, self).__init__()
        self.bert = bert_model
        self.lstm = nn.LSTM(bert_model.config.hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(hidden_size, num_classes)

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        # input_ids and attention_mask are lists of sentence embeddings and masks
        cls_embeddings = []

        for ids, mask in zip(input_ids, attention_mask):
            bert_output = self.bert(ids.unsqueeze(0), attention_mask=mask.unsqueeze(0))
            cls_embedding = bert_output.last_hidden_state[:, 0, :]
            cls_embeddings.append(cls_embedding)

        cls_embeddings = torch.cat(cls_embeddings, dim=0).unsqueeze(0)
        lstm_out, _ = self.lstm(cls_embeddings)
        out = self.dropout(lstm_out)
        logits = self.fc(out)
        return logits.squeeze(0)

# Load BERT model
bert_model = AutoModel.from_pretrained("bert-base-uncased")

# Instantiate the model
hidden_size = 128
num_classes = len(labels_emotions)  # Set this to the number of classes you have
model = BertLSTMClassifier(bert_model, hidden_size, num_classes, freeze_bert=True)


In [62]:
from transformers import AdamW
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)

for epoch in range(1):
    model.train()
    total_loss = 0

    for batch in train_dataloader:
        batch_input_ids, batch_attention_mask, batch_labels = tuple(t.to(device) for t in batch)

        model.zero_grad()

        outputs = model(batch_input_ids, batch_attention_mask)
        loss = torch.nn.functional.cross_entropy(outputs, batch_labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    avg_train_loss = total_loss / len(train_dataloader)
    print(f"Epoch {epoch}: Average training loss: {avg_train_loss}")

    # Validation step...


KeyboardInterrupt: ignored